In [138]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [139]:
torch.manual_seed(42)

In [140]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [141]:
df = pd.read_csv('fashion-mnist_train.csv')

In [142]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [143]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [144]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [145]:
X_train = X_train/255.0
X_test = X_test/255.0

In [146]:
#create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):

    return self.features[index], self.labels[index]


In [147]:
#create train_dataset object

train_dataset = CustomDataset(X_train,y_train)

In [148]:
#create test_dataset object
test_dataset = CustomDataset(X_test,y_test)

In [149]:
#create train and test loader
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

In [150]:
len(train_loader)

1500

In [151]:
class MyNN(nn.Module):

  def __init__(self,input_dim, output_dim, num_hidden_layers, neurons_per_layer):
    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(0.3))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self,x):
    return self.model(x)


In [152]:
#objective function

def objective(trial):

  #next hyperparameter values from search space

  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1 , 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8 , 128, step = 8)

  #model init

  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer)
  model.to(device)

  #params init

  learning_rate = 0.1
  epochs = 100

  #optimizer and loss selection
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = 1e-4)

  #training loop

  for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:
      #move to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        #forward pass
      outputs = model(batch_features)
        #calculate loss
      loss = criterion(outputs,batch_labels)
        #backprop
      optimizer.zero_grad()
      loss.backward()
        #update gradients
      optimizer.step()

  #evaluation

  model.eval()
  total = 0
  correct = 0

  with torch.no_grad():
    for batch_features, batch_labels in test_loader:

      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

  accuracy = correct / total
  print(f"Accuracy: {accuracy}")

  return accuracy

In [153]:
import optuna

study = optuna.create_study(direction = 'maximize' )

[I 2026-10-19 14:33:45,347] A new study created in memory with name: no-name-195ffe5a-5734-44c8-85f8-2843a83929fd


In [154]:
study.optimize(objective, n_trials = 10)

[I 2026-10-19 14:39:54,261] Trial 0 finished with value: 0.8580833333333333 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 32}. Best is trial 0 with value: 0.8580833333333333.


Accuracy: 0.8580833333333333


[I 2026-10-19 14:48:03,407] Trial 1 finished with value: 0.88975 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 88}. Best is trial 1 with value: 0.88975.


Accuracy: 0.88975


[I 2026-10-19 14:56:12,727] Trial 2 finished with value: 0.89275 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 128}. Best is trial 2 with value: 0.89275.


...

Accuracy: 0.89125


[I 2026-10-19 15:30:05,590] Trial 8 finished with value: 0.88825 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 96}. Best is trial 2 with value: 0.89275.


Accuracy: 0.88825


[I 2026-10-19 15:37:08,797] Trial 9 finished with value: 0.76375 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 16}. Best is trial 2 with value: 0.89275.


Accuracy: 0.76375
